# LSTM 데이터 생성

`orders.csv`와 `customer_features.csv`를 사용해 LSTM 학습용 시퀀스를 생성합니다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ORDERS_PATH = 'data/orders.csv'
FEATURE_PATH = 'data/customer_features.csv'
OUTPUT_DIR = 'outputs'
X_PATH = 'outputs/X_seq.npy'
Y_PATH = 'outputs/y.npy'
META_PATH = 'outputs/lstm_sequence_metadata.csv'

SEQ_LEN = 5
MAX_RECENT_ORDERS = 10
RANDOM_SEED = 42
DELAY_THRESHOLD = 15

In [2]:
def resolve_path(path_text: str) -> Path:
    # 경로 확인
    candidates = [Path(path_text), Path(Path(path_text).name)]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'파일을 찾을 수 없습니다: {path_text}')


def load_data():
    # 데이터 불러오기
    orders_file = resolve_path(ORDERS_PATH)
    feature_file = resolve_path(FEATURE_PATH)

    feature_cols = ['user_id', 'total_orders', 'avg_days_between_orders', 'max_order_number']
    features = pd.read_csv(feature_file, usecols=feature_cols)

    order_header = pd.read_csv(orders_file, nrows=0)
    order_cols = [
        col for col in ['user_id', 'order_number', 'days_since_prior_order', 'order_dow', 'order_hour_of_day']
        if col in order_header.columns
    ]
    required = {'user_id', 'order_number', 'days_since_prior_order'}
    missing = required - set(order_cols)
    if missing:
        raise ValueError(f'필수 컬럼이 없습니다: {sorted(missing)}')

    orders = pd.read_csv(orders_file, usecols=order_cols)

    print(f'[INFO] orders 크기: {orders.shape}')
    print(f'[INFO] features 크기: {features.shape}')
    return orders, features


def preprocess_data(orders: pd.DataFrame, features: pd.DataFrame):
    # 고객 필터링
    features = features.copy()
    for col in ['total_orders', 'avg_days_between_orders', 'max_order_number']:
        features[col] = features[col].fillna(features[col].median())

    threshold = features['total_orders'].quantile(0.8)
    features = features[features['total_orders'] >= threshold].drop_duplicates('user_id').copy()
    user_ids = set(features['user_id'].tolist())

    # 주문 전처리
    orders = orders[orders['user_id'].isin(user_ids)].copy()
    for col in ['order_dow', 'order_hour_of_day']:
        if col not in orders.columns:
            orders[col] = 0.0
    orders['days_since_prior_order'] = orders['days_since_prior_order'].fillna(0)
    orders['order_dow'] = orders['order_dow'].fillna(0)
    orders['order_hour_of_day'] = orders['order_hour_of_day'].fillna(0)
    orders = orders.sort_values(['user_id', 'order_number']).reset_index(drop=True)
    orders['target_label'] = (orders['days_since_prior_order'] > DELAY_THRESHOLD).astype(np.int8)

    user_max_order = orders.groupby('user_id')['order_number'].transform('max').clip(lower=1)
    orders['norm_order_number'] = (orders['order_number'] / user_max_order).astype(np.float32)
    orders['gap_change'] = orders.groupby('user_id')['days_since_prior_order'].diff().fillna(0).astype(np.float32)
    orders['gap_mean_3'] = (
        orders.groupby('user_id')['days_since_prior_order']
        .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
        .fillna(0)
        .astype(np.float32)
    )
    orders = orders.groupby('user_id', group_keys=False).tail(MAX_RECENT_ORDERS).reset_index(drop=True)

    valid_users = orders.groupby('user_id').size()
    valid_users = valid_users[valid_users >= SEQ_LEN + 1].index
    orders = orders[orders['user_id'].isin(valid_users)].reset_index(drop=True)
    features = features[features['user_id'].isin(valid_users)].copy()

    if orders.empty:
        raise ValueError('시퀀스를 만들 수 있는 사용자가 없습니다.')

    print(f'[INFO] 고빈도 고객 수: {features.user_id.nunique()}')
    print(f'[INFO] 전처리 후 주문 수: {len(orders)}')
    return orders, features


def build_sequences(orders: pd.DataFrame, features: pd.DataFrame):
    # 시퀀스 생성
    seq_cols = ['days_since_prior_order', 'order_number', 'norm_order_number', 'gap_change', 'gap_mean_3', 'order_dow', 'order_hour_of_day']
    static_cols = ['total_orders', 'avg_days_between_orders', 'max_order_number']
    static_map = features.set_index('user_id')[static_cols].astype(np.float32).to_dict('index')

    x_list = []
    y_list = []
    meta_rows = []

    for user_id, group in orders.groupby('user_id', sort=False):
        group = group.sort_values('order_number').reset_index(drop=True)
        seq_values = group[seq_cols].to_numpy(dtype=np.float32)
        static_values = np.array(list(static_map[user_id].values()), dtype=np.float32)
        static_values[0] = static_values[0] / max(static_values[0], 1.0)
        static_values[2] = static_values[2] / max(static_values[2], 1.0)

        for idx in range(SEQ_LEN, len(group)):
            seq_window = seq_values[idx - SEQ_LEN:idx]
            static_window = np.repeat(static_values.reshape(1, -1), SEQ_LEN, axis=0)
            x_list.append(np.concatenate([seq_window, static_window], axis=1))
            y_list.append(int(group.iloc[idx]['target_label']))
            meta_rows.append({
                'sample_id': len(meta_rows),
                'user_id': int(user_id),
                'target_order_number': int(group.iloc[idx]['order_number']),
                'target_label': int(group.iloc[idx]['target_label']),
            })

    if not x_list:
        raise ValueError('생성된 시퀀스가 없습니다.')

    x_seq = np.asarray(x_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.int8)
    meta = pd.DataFrame(meta_rows)

    print(f'[INFO] X_seq shape: {x_seq.shape}')
    print(f'[INFO] y shape: {y.shape}')
    print(f'[INFO] 라벨 분포: {np.bincount(y)}')
    return x_seq, y, meta


def save_outputs(x_seq: np.ndarray, y: np.ndarray, meta: pd.DataFrame):
    # 데이터 저장
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    np.save(X_PATH, x_seq)
    np.save(Y_PATH, y)
    meta.to_csv(META_PATH, index=False)
    print(f'[INFO] 저장: {X_PATH}')
    print(f'[INFO] 저장: {Y_PATH}')
    print(f'[INFO] 저장: {META_PATH}')

In [3]:
# 실행
orders, features = load_data()
orders, features = preprocess_data(orders, features)
X_seq, y, metadata = build_sequences(orders, features)
save_outputs(X_seq, y, metadata)

display(metadata.head())
display(pd.Series(y, name='target_label').value_counts().sort_index().to_frame())

[INFO] orders 크기: (3421083, 5)
[INFO] features 크기: (42499, 4)


[INFO] 고빈도 고객 수: 8778
[INFO] 전처리 후 주문 수: 87780


[INFO] X_seq shape: (43890, 5, 10)
[INFO] y shape: (43890,)
[INFO] 라벨 분포: [42464  1426]
[INFO] 저장: outputs/X_seq.npy
[INFO] 저장: outputs/y.npy
[INFO] 저장: outputs/lstm_sequence_metadata.csv


,sample_id,user_id,target_order_number,target_label
0,0,27,78,0
1,1,27,79,0
2,2,27,80,0
3,3,27,81,0
4,4,27,82,0


,count
target_label,
0,42464
1,1426
